# Master — MOZAK: NOVE LEZIJE (ISPRAVLJENO)

Treniran **samo na MSSEG-2**, a **open_ms_data  EKSTERNI test** (model ga nikad ne vidi).
- Ulaz: 2 kanala (FLAIR vreme1 + vreme2), poravnati.
- Trening: MSSEG-2 parovi.
- Eksterna evaluacija: open_ms_data longitudinalno (nezavisno).
- Interno (MSSEG-2 held-out, fold_0) za poređenje sa starim **0,362**.



## Ćelija 0 — priprema + metrike

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q nnunetv2 nibabel

import os, shutil, json, glob
import numpy as np, nibabel as nib
from scipy.ndimage import label

MASTER = '/content/drive/MyDrive/Master/mozak'
os.environ['nnUNet_raw']          = '/content/nnUNet_raw'
os.environ['nnUNet_preprocessed'] = '/content/nnUNet_preprocessed'
os.environ['nnUNet_results']      = f'{MASTER}/modeli/nnUNet_results'
for p in [os.environ['nnUNet_raw'], os.environ['nnUNet_preprocessed'], os.environ['nnUNet_results']]:
    os.makedirs(p, exist_ok=True)
import torch
print('GPU:', torch.cuda.is_available())
STRUKT = np.ones((3,3,3), int)
def ucitaj(p): return np.asarray(nib.load(p).dataobj)
def dice(pred, gt):
    p=pred>0; t=gt>0
    if p.sum()+t.sum()==0: return 1.0
    return float(2*(p&t).sum()/(p.sum()+t.sum()))
def lezijske(pred, gt):
    gl,ng = label(gt>0, STRUKT); pl,npr = label(pred>0, STRUKT)
    pm=pred>0; gm=gt>0
    tp = sum(1 for i in range(1,ng+1) if pm[gl==i].any())
    fp = sum(1 for j in range(1,npr+1) if not gm[pl==j].any())
    fn = ng-tp
    return dict(LTPR=tp/ng if ng else np.nan, LFPR=fp/npr if npr else 0.0,
                F1=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan)
print('spremno')

Mounted at /content/drive
GPU: True
spremno


## Ćelija 0b — save10 trener

In [ ]:
import os, torch, nnunetv2
vdir = os.path.join(os.path.dirname(nnunetv2.__file__),'training','nnUNetTrainer','variants','training_length')
with open(os.path.join(vdir,'nnUNetTrainer_100epochs_save10.py'),'w') as f:
    f.write("""import torch
from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import nnUNetTrainer_100epochs
class nnUNetTrainer_100epochs_save10(nnUNetTrainer_100epochs):
    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.save_every = 10
        self.initial_lr = 1e-3          # nizi LR -> stabilno, bez kolapsa na retkim ciljevima
        self.oversample_foreground_percent = 0.5
""")
print('trener: LR=1e-3, oversample=0.5, save_every=10')

trener: LR=1e-3, oversample=0.5, save_every=10


## Ćelija 1 — podaci: MSSEG-2 (trening) + open_ms (EKSTERNI test)

In [ ]:

MSSEG = None
for cand in ['/content/drive/MyDrive/training',
             '/content/drive/MyDrive/ms projekat/snimci_raspakovano/training']:
    if os.path.isdir(cand): MSSEG = cand; break
if MSSEG is None:
    hits = glob.glob('/content/drive/MyDrive/**/flair_time01_on_middle_space.nii.gz', recursive=True)
    if hits: MSSEG = os.path.dirname(os.path.dirname(hits[0]))
print('MSSEG-2 folder:', MSSEG)
assert MSSEG, 'MSSEG-2 nije nadjen! Podeli training folder sa drugim nalogom + dodaj shortcut.'

train_pairs = []
for pid in sorted(os.listdir(MSSEG)):
    d = f'{MSSEG}/{pid}'
    s1=f'{d}/flair_time01_on_middle_space.nii.gz'; s2=f'{d}/flair_time02_on_middle_space.nii.gz'; m=f'{d}/ground_truth.nii.gz'
    if all(os.path.exists(x) for x in (s1,s2,m)):
        train_pairs.append((f'msseg_{pid}', s1, s2, m))
print('MSSEG-2 (trening) parova:', len(train_pairs))

os.chdir('/content')
if not os.path.exists('/content/open_ms_data'):
    !git clone -q https://github.com/muschellij2/open_ms_data.git
oms = '/content/open_ms_data/longitudinal/coregistered'
oms_pairs = []
if os.path.exists(oms):
    for pac in sorted(os.listdir(oms)):
        d=f'{oms}/{pac}'
        s1=f'{d}/study1_FLAIR.nii.gz'; s2=f'{d}/study2_FLAIR.nii.gz'; m=f'{d}/gt.nii.gz'
        if os.path.isdir(d) and all(os.path.exists(x) for x in (s1,s2,m)):
            oms_pairs.append((f'oms_{pac}', s1, s2, m))
print('open_ms (EKSTERNI test) parova:', len(oms_pairs))
assert train_pairs and oms_pairs

MSSEG-2 folder: /content/drive/MyDrive/ms projekat/snimci_raspakovano/training
MSSEG-2 (trening) parova: 40
open_ms (EKSTERNI test) parova: 20


## Ćelija 2 — nnU-Net format, Dataset021_BrainNew

In [ ]:
DS = os.environ['nnUNet_raw'] + '/Dataset021_BrainNew'
shutil.rmtree(DS, ignore_errors=True)
for s in ['imagesTr','labelsTr']: os.makedirs(f'{DS}/{s}', exist_ok=True)
def upisi_masku(src, dst):
    img = nib.load(src)
    d = (np.asarray(img.dataobj) > 0).astype(np.uint8)
    nib.save(nib.Nifti1Image(d, img.affine, img.header), dst)

gt_msseg = {}
n = 0
for cid, s1, s2, m in train_pairs:
    shutil.copy(s1, f"{DS}/imagesTr/{cid}_0000.nii.gz")
    shutil.copy(s2, f"{DS}/imagesTr/{cid}_0001.nii.gz")
    upisi_masku(m, f"{DS}/labelsTr/{cid}.nii.gz")
    gt_msseg[cid] = m; n += 1
with open(f'{DS}/dataset.json','w') as f:
    json.dump({"channel_names":{"0":"FLAIR_time1","1":"FLAIR_time2"},
               "labels":{"background":0,"lesion":1},
               "numTraining":n,"file_ending":".nii.gz",
               "overwrite_image_reader_writer":"SimpleITKIO"}, f, indent=2)
print('upisano (MSSEG-2 trening):', n)

upisano (MSSEG-2 trening): 40


## Ćelija 3 — plan & preprocess (sa -np 2 da ne puca RAM)

In [ ]:
!nnUNetv2_plan_and_preprocess -d 21 --verify_dataset_integrity -npfp 2 -np 2

Fingerprint extraction...
Dataset021_BrainNew
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Extracting dataset fingerprint: 100% 40/40 [01:55<00:00,  2.89s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [1.00585938 1.00585938 0.54590111]. 
Current patch size: (np.int64(112), np.int64(128), np.int64(160)). 
Current median shape: [241.26213592 248.54368932 331.06796117]
Attempting to find 3d_lowres config. 
Curr

## Ćelija 4 — TRENING (GPU)

In [ ]:
rez = f"{os.environ['nnUNet_results']}/Dataset021_BrainNew/nnUNetTrainer_100epochs_save10__nnUNetPlans__3d_fullres/fold_0"
nastavi = "--c" if os.path.exists(f"{rez}/checkpoint_latest.pth") else ""
print("Nastavljam od checkpointa." if nastavi else "Krecem od nule.")
!nnUNetv2_train 21 3d_fullres 0 -tr nnUNetTrainer_100epochs_save10 {nastavi}

Nastavljam od checkpointa.

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-11-09 11:12:07.953045: Using torch.compile...
2026-11-09 11:12:25.612427: do_dummy_2d_data_a

## Ćelija 5 — EVALUACIJA: EKSTERNO (open_ms) + interno (MSSEG-2 )

In [ ]:
TR = f"{os.environ['nnUNet_results']}/Dataset021_BrainNew/nnUNetTrainer_100epochs_save10__nnUNetPlans__3d_fullres"

in_e='/content/eval_oms/in'; out_e='/content/eval_oms/out'
os.makedirs(in_e, exist_ok=True)
for cid,s1,s2,m in oms_pairs:
    shutil.copy(s1, f'{in_e}/{cid}_0000.nii.gz'); shutil.copy(s2, f'{in_e}/{cid}_0001.nii.gz')
!nnUNetv2_predict -i {in_e} -o {out_e} -d 21 -c 3d_fullres -f 0 -tr nnUNetTrainer_100epochs_save10 -chk checkpoint_best.pth

ext=[]
for cid,s1,s2,m in oms_pairs:
    pp=f'{out_e}/{cid}.nii.gz'
    if not os.path.exists(pp): continue
    pr=ucitaj(pp); gt=(ucitaj(m)>0)
    ext.append((dice(pr,gt), lezijske(pr,gt)))
if ext:
    print(f"\n=== EKSTERNO (open_ms, {len(ext)} parova, NEZAVISNO) ===")
    print(f"  Dice={np.nanmean([d for d,_ in ext]):.3f}  LTPR={np.nanmean([L['LTPR'] for _,L in ext]):.3f}  "
          f"LFPR={np.nanmean([L['LFPR'] for _,L in ext]):.3f}")

val = f"{TR}/fold_0/validation"
if os.path.exists(val):
    iv=[]
    for f in sorted(os.listdir(val)):
        if not f.endswith('.nii.gz'): continue
        cid=f.replace('.nii.gz','')
        if cid not in gt_msseg: continue
        pr=ucitaj(f'{val}/{f}'); gt=(ucitaj(gt_msseg[cid])>0)
        iv.append(dice(pr,gt))
    print(f"\n=== INTERNO (MSSEG-2 held-out, {len(iv)} parova) ===")
    print(f"  Dice={np.nanmean(iv):.3f}   (stari rucni model MSSEG-2: 0.362)")

rez_dir=f'{MASTER}/rezultati'; os.makedirs(rez_dir, exist_ok=True)
print('\n(open_ms je cist nezavisan rezultat; MSSEG-2 held-out za poredjenje sa 0.362)')


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
There are 20 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 20 cases that I would like to predict

Predicting oms_patient01:
perform_everything_on_device: True
100% 24/24 [00:29<00:00,  1.25s/it]
sending off prediction to background worker for resampling and export
done with oms_patient01

Predicting oms_patient02:
perform_everything_on_device: True
100% 24/24